In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/drug_reviews_clean_20251102_0952.csv')

In [ ]:
df.columns

Index(['source', 'drug_name', 'drug_review', 'age', 'gender', 'weight'], dtype='object')

In [ ]:
df['age'] = df['age'].fillna(df['age'].median())
df['weight'] = df['weight'].fillna(df['weight'].median())

In [ ]:
import numpy as np

# Define possible gender values
gender_choices = ["Male", "Female"]

# Fill missing gender with random choice
df['gender'] = df['gender'].apply(
    lambda x: np.random.choice(gender_choices) if pd.isna(x) else x
)

df.to_csv("drug_reviews_imputed.csv", index=False)

In [ ]:
df.head()

,source,drug_name,drug_review,age,gender,weight
0,drugs_com,metformin,I gained A LOT of weight and had unwanted hair...,30.0,Male,5.9
1,drugs_com,metformin,"""On Metformin for nearly two years. In Canadia...",30.0,Female,11.3
2,drugs_com,metformin,Reviews and ratings for Metformin when used in...,30.0,Female,13.6
3,drugs_com,metformin,Reviews and ratings for Empagliflozin/metformi...,30.0,Female,13.6
4,drugs_com,metformin,Main side effect is extreme tiredness and fati...,30.0,Female,13.6


In [ ]:
df.isnull().sum()

,0
source,0
drug_name,0
drug_review,0
age,0
gender,0
weight,0


In [ ]:
df.head()

,source,drug_name,drug_review,age,gender,weight
0,drugs_com,metformin,I gained A LOT of weight and had unwanted hair...,30.0,Male,5.9
1,drugs_com,metformin,"""On Metformin for nearly two years. In Canadia...",30.0,Female,11.3
2,drugs_com,metformin,Reviews and ratings for Metformin when used in...,30.0,Female,13.6
3,drugs_com,metformin,Reviews and ratings for Empagliflozin/metformi...,30.0,Female,13.6
4,drugs_com,metformin,Main side effect is extreme tiredness and fati...,30.0,Female,13.6


In [ ]:
import pandas as pd
df.to_csv('mtech_dataset.csv', index=False)
from google.colab import files
files.download('mtech_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# -----------------------
# Load
# -----------------------
df = pd.read_csv("/content/drug_reviews_clean_20251102_0952.csv")

# Quick copy for safety
df_out = df.copy()

# -----------------------
# 1) Create numeric predictor features (no missing ideally)
# -----------------------
# review length (number of chars) - if review is NaN treat as 0
df_out['review_len'] = df_out['drug_review'].fillna("").str.len()

# drug_name frequency (how many times this drug appears) - numeric
drug_freq = df_out['drug_name'].value_counts()
df_out['drug_freq'] = df_out['drug_name'].map(drug_freq).fillna(0)

# source frequency (if you have multiple sources)
source_freq = df_out['source'].value_counts()
df_out['source_freq'] = df_out['source'].map(source_freq).fillna(0)

# -----------------------
# 2) Build feature frame for imputation
# We include age, weight (with NaNs) plus predictors (no NaNs)
# -----------------------
features = df_out[['age', 'weight', 'review_len', 'drug_freq', 'source_freq']].copy()

# Convert empty strings to NaN for numeric cols just in case
features['age'] = pd.to_numeric(features['age'], errors='coerce')
features['weight'] = pd.to_numeric(features['weight'], errors='coerce')

# -----------------------
# 3) Standardize each column using mean/std computed ignoring NaNs
# This ensures KNN distance is reasonable across columns
# -----------------------
col_means = features.mean(skipna=True)
col_stds = features.std(skipna=True).replace(0, 1)  # avoid division by zero

features_scaled = (features - col_means) / col_stds

# -----------------------
# 4) KNN Imputation on scaled features
# -----------------------
imputer = KNNImputer(n_neighbors=5, weights="uniform", metric="nan_euclidean")
imputed_array = imputer.fit_transform(features_scaled)

# imputed_array columns follow the same order: ['age','weight','review_len','drug_freq','source_freq']
imputed_scaled = pd.DataFrame(imputed_array, columns=features_scaled.columns, index=features_scaled.index)

# -----------------------
# 5) Convert imputed scaled age/weight back to original scale
# -----------------------
imputed = imputed_scaled * col_stds + col_means

# Replace original age/weight with imputed values where they were missing
# For age: round to nearest integer (adult ages generally integer)
age_mask = df_out['age'].isna()
df_out.loc[age_mask, 'age'] = imputed.loc[age_mask, 'age'].round().astype(int)

# For weight: keep one decimal place and clip to a reasonable range (30 - 200 kg)
weight_mask = df_out['weight'].isna()
df_out.loc[weight_mask, 'weight'] = imputed.loc[weight_mask, 'weight'].round(1)
df_out['weight'] = pd.to_numeric(df_out['weight'], errors='coerce')

# Clip unrealistic weights (optionally adjust bounds)
df_out.loc[df_out['weight'] < 30, 'weight'] = 30.0
df_out.loc[df_out['weight'] > 200, 'weight'] = 200.0

# -----------------------
# 6) Random gender imputation (Male / Female) using dataset distribution
# You asked for no 'Unknown' — only Male/Female
# -----------------------
# Compute current distribution among non-missing values
existing_dist = df_out['gender'].value_counts(normalize=True)

# fallback to 50/50 if neither present
p_male = existing_dist.get("Male", 0.5)
p_female = existing_dist.get("Female", 0.5)

# If the column contains other values (like 'M'/'F' etc), normalize them first:
# (Optional step) Normalize common variants to 'Male'/'Female' before sampling:
df_out['gender'] = df_out['gender'].replace({'M':'Male', 'F':'Female', 'm':'Male', 'f':'Female'})

# Recompute distribution after normalization
existing_dist = df_out['gender'].value_counts(normalize=True)
p_male = existing_dist.get("Male", p_male)
p_female = existing_dist.get("Female", p_female)

# Apply random assignment only where gender is missing / NaN
mask_gender_missing = df_out['gender'].isna()
n_missing = mask_gender_missing.sum()
if n_missing > 0:
    df_out.loc[mask_gender_missing, 'gender'] = np.random.choice(
        ["Male", "Female"], size=n_missing, p=[p_male, p_female]
    )

# -----------------------
# 7) Final safety checks & Save
# -----------------------
# Count remaining nulls (should be zero for age/weight/gender)
print("Null counts after imputation:\n", df_out[['age','weight','gender']].isna().sum())

# Quick sanity: show some problematic values if any
print("Age min/max:", df_out['age'].min(), df_out['age'].max())
print("Weight min/max:", df_out['weight'].min(), df_out['weight'].max())
print("Gender value counts:\n", df_out['gender'].value_counts())

# Save
df_out.to_csv("drug_reviews_imputed_knn.csv", index=False)
print("Saved imputed dataset to drug_reviews_imputed_knn.csv")


Null counts after imputation:
 age       0
weight    0
gender    0
dtype: int64
Age min/max: 10.0 100.0
Weight min/max: 30.0 199.6
Gender value counts:
 gender
Male      17004
Female    16959
male      10996
female      621
Name: count, dtype: int64
Saved imputed dataset to drug_reviews_imputed_knn.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor

# Load dataset
df = pd.read_csv("/content/drug_reviews_clean_20251102_0952.csv")

df_out = df.copy()

# -----------------------------------------
# 1. Create numeric predictors (good for RF)
# -----------------------------------------
df_out['review_len'] = df_out['drug_review'].fillna("").str.len()

drug_freq = df_out['drug_name'].value_counts()
df_out['drug_freq'] = df_out['drug_name'].map(drug_freq).fillna(0)

source_freq = df_out['source'].value_counts()
df_out['source_freq'] = df_out['source'].map(source_freq).fillna(0)

# -----------------------------------------
# 2. Prepare RF Imputer for numeric columns
# -----------------------------------------
numeric_cols = ['age', 'weight', 'review_len', 'drug_freq', 'source_freq']

imputer = IterativeImputer(
    estimator=RandomForestRegressor(
        n_estimators=50,
        random_state=42,
        n_jobs=-1
    ),
    max_iter=10,
    random_state=42
)

# Apply imputer ONLY to numeric columns
numeric_imputed = imputer.fit_transform(df_out[numeric_cols])

# Create a dataframe
numeric_imputed_df = pd.DataFrame(numeric_imputed, columns=numeric_cols)

# Replace age and weight only
df_out['age'] = numeric_imputed_df['age'].round().astype(int)
df_out['weight'] = numeric_imputed_df['weight'].round(1)

# Optional clipping for realistic ranges
df_out['age'] = df_out['age'].clip(lower=10, upper=100)
df_out['weight'] = df_out['weight'].clip(lower=30, upper=200)

# -----------------------------------------
# 3. Random Gender Imputation (Male/Female)
# -----------------------------------------
df_out['gender'] = df_out['gender'].replace({
    'M': 'Male', 'F': 'Female', 'm': 'Male', 'f': 'Female'
})

gender_dist = df_out['gender'].value_counts(normalize=True)

p_male = gender_dist.get("Male", 0.5)
p_female = gender_dist.get("Female", 0.5)

mask = df_out['gender'].isna()
df_out.loc[mask, 'gender'] = np.random.choice(
    ["Male", "Female"],
    size=mask.sum(),
    p=[p_male, p_female]
)

# -----------------------------------------
# 4. Save Output
# -----------------------------------------
df_out.to_csv("drug_reviews_imputed_rf.csv", index=False)

print("Random Forest imputation completed. Saved to drug_reviews_imputed_rf.csv")


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


Random Forest imputation completed. Saved to drug_reviews_imputed_rf.csv
